<a href="https://colab.research.google.com/github/frandresst-wq/Proyecto-de-Ciencia-de-Datos/blob/main/S2_Notebook_Laboratorio_CDD_Soto_Franco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | _(escribe tu nombre aquí)_ |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [ ]:
import pandas as pd
import numpy as np

RUTA = ''
SEMILLA = 20262
np.random.seed(SEMILLA)

ARCHIVO = 'S1_Dataset_Guia_CDD.csv'   # <- reemplaza
SEPARADOR = ';'                        # <- ajusta si corresponde

df = pd.read_csv(RUTA + ARCHIVO, sep=SEPARADOR)
print('Forma:', df.shape)
df.head(200)

Forma: (734, 11)


,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado
0,02-20260626,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-06-26,77.1,154.9,9.5,58.3,SI
1,08-20260405,8,Coronel Norte,Coronel,Biobío,2026-04-05,31.6,52.0,12.7,65.4,SI
2,06-20260621,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-06-21,63.5,111.4,6.7,83.0,SI
3,01-20260531,1,Kingston,Concepción,Biobío,2026-05-31,48.1,79.4,14.4,75.1,SI
4,05-20260522,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-05-22,85.0,162.1,11.5,82.9,SI
...,...,...,...,...,...,...,...,...,...,...,...
195,07-20260619,7,Valdivia Centro,Valdivia,Los Ríos,2026-06-19,64.0,130.3,9.3,83.8,SI
196,06-20260531,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-05-31,47.5,98.9,10.4,99.0,SI
197,04-20260614,4,Talcahuano,Talcahuano,Biobío,2026-06-14,29.1,52.8,7.3,77.7,SI
198,05-20260407,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-04-07,72.0,144.0,19.5,78.2,SI


In [ ]:
df.dtypes

,0
id_registro,object
estacion_id,int64
estacion,object
comuna,object
region,object
fecha,object
pm25,float64
pm10,float64
temperatura_c,object
humedad_pct,float64


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

++++++++ ¿Cómo evaluar estas dimensiones con Pandas?

1. **Completitud:**
Por ejemplo, df.isna().sum()
Cuenta los valores faltantes de cada columna.

2. **Exactitud:**
Por ejemplo, df.describe().T.round(2)
Permite identificar valores mínimos, máximos y rangos sospechosos en variables numéricas.

3. **Consistencia:**
Por ejemplo, (df['fecha_termino'] < df['fecha_inicio']).sum()
Cuenta los registros cuya fecha de término es anterior a la fecha de inicio, suponiendo que ambas columnas contienen fechas válidas.

4. **Unicidad:**
Por ejemplo, df.duplicated().sum()
Cuenta las filas duplicadas exactas, sin incluir la primera aparición.

5. **Oportunidad:**
Por ejemplo, df['fecha'].max()
Obtiene la fecha más reciente disponible en el conjunto de datos, siempre que la columna tenga un tipo de fecha adecuado.


### 1.1 Completitud

In [ ]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
pm25,35,4.77
estacion_id,0,0.00
id_registro,0,0.00
estacion,0,0.00
comuna,0,0.00
region,0,0.00
fecha,0,0.00
pm10,0,0.00
temperatura_c,0,0.00
humedad_pct,0,0.00


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [ ]:
# Tu código aquí
tabla = pd.crosstab(
    df['fecha'],  #cruzar con fecha o estacion
    df['pm25'].isna()
)

tabla.columns = ['Presentes', 'Faltantes']

tabla['Porcentaje'] = (
    tabla['Faltantes'] / tabla.sum(axis=1) * 100
).round(2)

tabla = tabla.sort_values('Faltantes', ascending=False)

display(tabla)

,Presentes,Faltantes,Porcentaje
fecha,,,
2026-05-01,6,2,25.00
2026-05-17,7,2,22.22
2026-06-04,5,2,28.57
2026-06-28,6,2,25.00
2026-06-27,5,2,28.57
...,...,...,...
25/04/2026,1,0,0.00
25/06/2026,1,0,0.00
26/04/2026,1,0,0.00


In [ ]:
df['fecha'] = pd.to_datetime(
    df['fecha'], format='mixed', dayfirst=True
)

# Cruzar por mes
tabla = pd.crosstab(
    df['fecha'].dt.to_period('M'),
    df['pm25'].isna()
)

tabla = tabla.reindex(columns=[False, True], fill_value=0)
tabla.columns = ['Presentes', 'Faltantes']

tabla['Porcentaje'] = (
    tabla['Faltantes'] / tabla.sum(axis=1) * 100
).round(2)

tabla = tabla.sort_values('Faltantes', ascending=False)

display(tabla)

,Presentes,Faltantes,Porcentaje
fecha,,,
2026-05,239,13,5.16
2026-06,221,12,5.15
2026-04,239,10,4.02


++++++ No se observa un patrón temporal evidente, ya que los porcentajes son similares entre los tres meses.

++++++ Sin embargo, el análisis anterior por estación mostró diferencias más importantes, lo que sugiere una posible concentración de faltantes según la estación de medición.
Se recomienda investigar sus causas antes de decidir si corresponde imputar los valores o declararlos como una limitación.

### 1.2 Exactitud — rangos y valores imposibles

In [ ]:
df.describe().T.round(2)

,count,mean,min,25%,50%,75%,max,std
estacion_id,734.0,4.491826,1.0,2.25,4.0,6.0,8.0,2.286281
fecha,734,2026-05-15 04:42:30.408719360,2026-04-01 00:00:00,2026-04-23 00:00:00,2026-05-15 00:00:00,2026-06-07 00:00:00,2026-06-29 00:00:00,NaN
pm25,699.0,62.039628,12.9,46.2,57.2,67.1,586.7,50.804083
pm10,734.0,77.099046,-999.0,81.675,103.8,122.1,201.4,173.30003
humedad_pct,734.0,75.850954,50.7,70.2,75.7,81.8,99.0,8.633779


++++++++ Viendo **este** describe en pm10 hay un posible valor centinela -999

### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [ ]:
# Tu código aquí. Sugerencia:
# for col in df.select_dtypes(include='number').columns:
#     print(col, sorted(df[col].dropna().unique())[:5], '...', sorted(df[col].dropna().unique())[-5:])


In [ ]:
# Buscar valores centinela
centinelas = [-1, 0, 999, 9999, -999]

for col in df.select_dtypes(include='number').columns:
    print('\nColumna:', col)
    for valor in centinelas:
        cantidad = (df[col] == valor).sum()
        if cantidad > 0:
            print('Valor:', valor, '- Cantidad:', cantidad)


Columna: estacion_id

Columna: pm25

Columna: pm10
Valor: -999 - Cantidad: 18

Columna: humedad_pct


+++++++ El valor -999 no representa una medición válida de material particulado, ya que las concentraciones de PM10 no pueden ser negativas.

Por lo tanto, estos 18 registros probablemente corresponden a mediciones ausentes o inválidas.
¿Qué debemos hacer?
Reemplazar los valores -999 por NaN para reconocerlos como faltantes

### 1.3 Consistencia — reglas cruzadas

+++++++ La consistencia es una dimensión de la calidad de datos que evalúa si la información mantiene coherencia entre sus valores, columnas, registros y reglas del conjunto de datos.

En otras palabras, busca detectar datos que se contradicen entre sí.

Ejemplos:\
Un trabajador tiene 15 años y registra 20 años de experiencia.\
Una estación pertenece a la comuna de Concepción en un registro y a Valdivia en otro.\
Una fecha de término es anterior a la fecha de inicio.\
Un producto tiene precio total de $50.000, pero cantidad 10 y precio unitario $8.000.

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [ ]:
# Regla 1:
# Una estación debe pertenecer siempre a la misma comuna
# Una estación de monitoreo no debería aparecer asociada a distintas comunas.
tabla = df.groupby('estacion')['comuna'].nunique()

print('Estaciones con comunas inconsistentes:')
print(tabla[tabla > 1])
df.groupby('estacion')['comuna'].unique()



Estaciones con comunas inconsistentes:
estacion
Coronel Norte             2
Coyhanco                  2
Kingston                  2
Los Ángeles Oriente       2
Osorno Entre Lagos        2
Talcahuano                2
Temuco Padre Las Casas    2
Valdivia Centro           2
Name: comuna, dtype: int64


,comuna
estacion,
Coronel Norte,"[Coronel, CORONEL ]"
Coyhanco,"[Chillán, CHILLÁN ]"
Kingston,"[Concepción, CONCEPCIÓN ]"
Los Ángeles Oriente,"[Los Ángeles, LOS ÁNGELES ]"
Osorno Entre Lagos,"[Osorno, OSORNO ]"
Talcahuano,"[Talcahuano, TALCAHUANO ]"
Temuco Padre Las Casas,"[Temuco, TEMUCO ]"
Valdivia Centro,"[Valdivia, VALDIVIA ]"


+++++++ Las 8 estaciones presentan dos formas de escribir la misma comuna: una con mayúscula inicial y otra completamente en mayúsculas.\
Se deben normalizar los nombres, unificar su escritura, de manera que una misma comuna siempre tenga el mismo nombre.\
Por ejemplo, eliminar espacios sobrantes, convertir a Mayúscual sólo la primera letra.

In [ ]:
# Regla 2:
# Cada identificador de estación debe corresponder a una única estación
# Un mismo estacion_id no debería identificar estaciones diferentes.
tabla = df.groupby('estacion_id')['estacion'].nunique()

print('Identificadores inconsistentes:')
print(tabla[tabla > 1])

Identificadores inconsistentes:
Series([], Name: estacion, dtype: int64)


+++++++ Esto significa que no se encontraron inconsistencias.

La serie está vacía porque ningún estacion_id está asociado a más de una estación.

Por lo tanto, la Regla 2 se cumple.

### 1.4 Unicidad

In [ ]:
print('Duplicados exactos:', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
# print('ID repetidos:', df['TU_ID'].duplicated().sum())

Duplicados exactos: 14


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

+++++++ En este conjunto de datos de calidad del aire, podemos utilizar las columnas:
- estacion: identifica la estación de monitoreo.
- fecha: identifica el día de la medición.\
La combinación estación + fecha puede utilizarse como clave, suponiendo que existe una sola medición diaria por estación.

In [ ]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Tu código aquí
# Crear clave normalizada
df['clave'] = (
    normalizar_texto(df['estacion']) + '_' +
    df['fecha'].astype(str).str[:10]
)
# Contar registros con claves repetidas
print('Casi-duplicados:', df['clave'].duplicated().sum())

Casi-duplicados: 14


In [ ]:
#Cuales son?
duplicados = df[df['clave'].duplicated(keep=False)]
display(duplicados.sort_values('clave'))

,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado,clave
98,01-20260405,1,Kingston,Concepción,Biobío,2026-04-05,19.9,36.1,13.7,65.8,SI,KINGSTON_2026-04-05
200,01-20260405,1,Kingston,Concepción,Biobío,2026-04-05,19.9,36.1,13.7,65.8,SI,KINGSTON_2026-04-05
488,01-20260629,1,Kingston,Concepción,Biobío,2026-06-29,59.5,100.0,10.9,86.8,SI,KINGSTON_2026-06-29
161,01-20260629,1,Kingston,Concepción,Biobío,2026-06-29,59.5,100.0,10.9,86.8,SI,KINGSTON_2026-06-29
405,02-20260412,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-04-12,55.0,100.7,12.5,77.8,SI,LOS ANGELES ORIENTE_2026-04-12
249,02-20260412,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-04-12,55.0,100.7,12.5,77.8,SI,LOS ANGELES ORIENTE_2026-04-12
610,02-20260520,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-05-20,72.2,127.4,7.6,73.6,SI,LOS ANGELES ORIENTE_2026-05-20
119,02-20260520,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-05-20,72.2,127.4,7.6,73.6,SI,LOS ANGELES ORIENTE_2026-05-20
335,06-20260404,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-04-04,62.7,131.4,15.6,64.3,SI,OSORNO ENTRE LAGOS_2026-04-04
470,06-20260404,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-04-04,62.7,131.4,15.6,64.3,SI,OSORNO ENTRE LAGOS_2026-04-04


++++++ Antes de eliminar registros, verifica que tu conjunto contenga una sola medición por estación y día. Si existen varias mediciones diarias legítimas, necesitarías incluir también la hora u otro identificador en la clave.

### 1.5 Oportunidad

In [ ]:
# Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?
# ¿El conjunto está vigente para la pregunta que quieres responder?


+++++++ La oportunidad es la dimensión de calidad que evalúa si los datos están disponibles y actualizados en el momento en que se necesitan para realizar un análisis o tomar decisiones.\
¿Los datos llegan a tiempo y están suficientemente actualizados para ser útiles?\


In [ ]:
#Fechas más recientes
print('Primera medición:', df['fecha'].min())
print('Última medición:', df['fecha'].max())
print('Fecha actual:', pd.Timestamp.today().date())

Primera medición: 2026-04-01 00:00:00
Última medición: 2026-06-29 00:00:00
Fecha actual: 2026-10-23


In [ ]:
#Cuántos días han pasado desde la última medición
ultima_fecha = df['fecha'].max()
hoy = pd.Timestamp.today().normalize()

dias = (hoy - ultima_fecha).days

print('Días desde la última medición:', dias)

Días desde la última medición: 116


++++++++ Este conjunto contiene mediciones de pm25, pm10, temperatura y humedad entre abril y junio de 2026.\
Por lo tanto, estos datos pueden ser adecuados para un análisis histórico, pero no para conocer la contaminación actual\
Un conjunto de datos puede tener buena completitud, exactitud y consistencia, pero baja oportunidad si la información llega demasiado tarde para ser utilizada.

---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

+++++ En la Parte 1 se detectaron 18 valores -999 en la columna pm10. Ahora corresponde corregirlos, reemplazándolos por NaN, ya que una concentración de material particulado no puede ser negativa.

In [ ]:
# Tu código aquí
# Reemplazar valores centinela por NaN
df['pm10'] = df['pm10'].replace(-999, np.nan)

# Verificar los faltantes
print(df[['pm10', 'pm25']].isna().sum())

pm10    18
pm25    35
dtype: int64


### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

+++++++  En este conjunto de datos de calidad del aire, podemos normalizar las columnas categóricas **estacion, comuna, region y validado**.

El objetivo es unificar las categorías que representan lo mismo, pero están escritas de manera diferente.

In [ ]:
# Tu código aquí
# Normalización Mecánica
# Se eliminan espacios, se unifican mayúsculas y se quitan tildes.
# Normalizar categorías
columnas = ['estacion', 'comuna', 'region', 'validado']

for col in columnas:
    df[col] = normalizar_texto(df[col]) #la función normalizar_texto está arriba, en la actividad 1.4

# Revisar las categorías resultantes
for col in columnas:
    print('\nColumna:', col)
    print(df[col].unique())


Columna: estacion
['LOS ANGELES ORIENTE' 'CORONEL NORTE' 'OSORNO ENTRE LAGOS' 'KINGSTON'
 'TEMUCO PADRE LAS CASAS' 'TALCAHUANO' 'COYHANCO' 'VALDIVIA CENTRO']

Columna: comuna
['LOS ANGELES' 'CORONEL' 'OSORNO' 'CONCEPCION' 'TEMUCO' 'TALCAHUANO'
 'CHILLAN' 'VALDIVIA']

Columna: region
['BIOBIO' 'LOS LAGOS' 'ARAUCANIA' 'NUBLE' 'LOS RIOS']

Columna: validado
['SI']


In [ ]:
# Categorías que quedan
for col in columnas:
    print('\n',col, ':', df[col].nunique(), 'categorías')
    print(df[col].value_counts())


 estacion : 8 categorías
estacion
TALCAHUANO                94
OSORNO ENTRE LAGOS        93
KINGSTON                  92
LOS ANGELES ORIENTE       92
TEMUCO PADRE LAS CASAS    92
VALDIVIA CENTRO           91
CORONEL NORTE             90
COYHANCO                  90
Name: count, dtype: int64

 comuna : 8 categorías
comuna
TALCAHUANO     94
OSORNO         93
CONCEPCION     92
LOS ANGELES    92
TEMUCO         92
VALDIVIA       91
CORONEL        90
CHILLAN        90
Name: count, dtype: int64

 region : 5 categorías
region
BIOBIO       368
LOS LAGOS     93
ARAUCANIA     92
LOS RIOS      91
NUBLE         90
Name: count, dtype: int64

 validado : 1 categorías
validado
SI    734
Name: count, dtype: int64


++++++ La categoría validado no se pudo normalizar mecánicamente, porque son cadenas distintas: S, N, SI, NO\
Se crea un diccionario de equivalencia

In [ ]:
equivalencias = {
    'S': 'SI',
    'N': 'NO'
}

df['validado'] = df['validado'].replace(equivalencias)
# Mostrar categorías y cantidades
print(df['validado'].value_counts())

print(df['validado'].unique())

display(df['validado'].value_counts().to_frame('Cantidad'))

validado
SI    734
Name: count, dtype: int64
['SI']


,Cantidad
validado,
SI,734


### ✏️ Actividad 2.3 — Duplicados

+++++  En la Parte 1 se detectaron 14 registros duplicados exactos en el conjunto de datos de calidad del aire.\
Ahora se debe decidir si corresponde corregir, descartar o declarar como limitación.\
En este caso, podemos descartar los duplicados exactos, ya que contienen la misma información y mantenerlos podría alterar los resultados del análisis.

In [ ]:
# Tu código aquí
# Código para eliminar duplicados
# Contar duplicados antes
print('Duplicados antes:', df.duplicated().sum())

# Eliminar duplicados exactos
df = df.drop_duplicates().copy()

# Verificar después
print('Duplicados después:', df.duplicated().sum())

# Mostrar cantidad de registros
print('Registros restantes:', len(df))

Duplicados antes: 14
Duplicados después: 0
Registros restantes: 720


In [ ]:
duplicados = df[df.duplicated(
    subset=['estacion', 'fecha'],
    keep=False
)]

display(duplicados.sort_values(['estacion', 'fecha']))

,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado,clave


In [ ]:
print('Duplicados exactos:', df.duplicated().sum())

print('Duplicados por estación y fecha:',
      df.duplicated(subset=['estacion', 'fecha']).sum())

Duplicados exactos: 0
Duplicados por estación y fecha: 0


+++++++++ Al verificar la unicidad mediante la combinación de estación y fecha, no se encontraron registros duplicados en el conjunto de datos actual.\
Esto indica que cada estación presenta un único registro por fecha. Por lo tanto, no es necesario eliminar registros adicionales según este criterio.

### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

+++++ En este conjunto de datos de calidad del aire, se debe verificar que las columnas numéricas sean realmente numéricas y que fecha tenga el tipo datetime.

In [ ]:
# Tu código aquí
# Convertir columnas numéricas
columnas = ['pm25', 'pm10', 'temperatura_c', 'humedad_pct']
for col in columnas:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.'),
        errors='coerce'
    )

# Convertir fechas con formatos mixtos
fechas = df['fecha'].astype(str).str.strip()
fecha1 = pd.to_datetime(
    fechas, format='%Y-%m-%d', errors='coerce'
)

fecha2 = pd.to_datetime(
    fechas, format='%d/%m/%Y', errors='coerce'
)
df['fecha'] = fecha1.fillna(fecha2)

# Mostrar tipos de datos
print(df.dtypes)

id_registro              object
estacion_id               int64
estacion                 object
comuna                   object
region                   object
fecha            datetime64[ns]
pm25                    float64
pm10                    float64
temperatura_c           float64
humedad_pct             float64
validado                 object
clave                    object
dtype: object


In [ ]:
# comprobar si alguna conversión produjo valores NaN adicionales
print( df.isna().sum())

id_registro       0
estacion_id       0
estacion          0
comuna            0
region            0
fecha             0
pm25             34
pm10             18
temperatura_c     0
humedad_pct       0
validado          0
clave             0
dtype: int64


++++++  Después de realizar las transformaciones de tipos de datos, el conjunto presenta 34 valores faltantes en pm25 y 18 en pm10, mientras que las demás columnas no presentan ausencias.\
La disminución de 35 a 34 faltantes en pm25 podría deberse a la eliminación de registros duplicados realizada anteriormente.

### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

++++++ En este conjunto de datos de calidad del aire, se deben identificar valores extremos en pm25, pm10, temperatura_c y humedad_pct, y determinar si podrían ser errores de registro o fenómenos reales.\
Anteriormente se encontró un valor máximo de 586,7 en pm25, que merece una revisión especial.

++++++ Se pueden identificar los valores atípicos con el método IQR. el cual identifica valores que se alejan considerablemente del rango central de los datos (un valor se considera atípico si queda muy por debajo o por encima del rango donde se concentra la mayoría de los datos).\
Q1 (primer cuartil, 25 %): el valor por debajo del cual se encuentra aproximadamente el 25 % de los datos.\
Q3 (tercer cuartil, 75 %): el valor por debajo del cual se encuentra aproximadamente el 75 % de los datos.\
El **rango intercuartílico** (Interquartile Range, IQR),  representa la amplitud del 50 % central de los datos. Al usar el tramo entre Q1 y Q3, el IQR es menos sensible a valores extremos que otras medidas, como el rango total.\
Es posible definir los **límites** para identificar posibles atípicos:\
Límite inferior: valores menores que Q1 - 1.5 × IQR.\
Límite superior: valores mayores que Q3 + 1.5 × IQR.\
El factor 1.5 es una convención habitual del método de Tukey. Es una regla estadística práctica: señala observaciones inusualmente alejadas del centro, pero no demuestra por sí sola que sean errores.

In [ ]:
# Tu código aquí
# Identificar valores atípicos
columnas = ['pm25', 'pm10', 'temperatura_c', 'humedad_pct']

for col in columnas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    inferior = Q1 - 1.5 * IQR
    superior = Q3 + 1.5 * IQR

    atipicos = df[
        (df[col] < inferior) | (df[col] > superior)
    ]

    print('\nVariable:', col)
    print('Límite inferior:', round(inferior, 2))
    print('Límite superior:', round(superior, 2))
    print('Cantidad de atípicos:', len(atipicos))


Variable: pm25
Límite inferior: 15.05
Límite superior: 98.45
Cantidad de atípicos: 16

Variable: pm10
Límite inferior: 26.79
Límite superior: 179.89
Cantidad de atípicos: 5

Variable: temperatura_c
Límite inferior: 2.45
Límite superior: 19.65
Cantidad de atípicos: 3

Variable: humedad_pct
Límite inferior: 52.95
Límite superior: 99.15
Cantidad de atípicos: 1


+++++++ Cruzar los valores extremos con estación y fecha.\
Como el ejercicio pide comprobar su plausibilidad, podemos revisar las mediciones extremas de pm25 junto con la estación, fecha y pm10.

In [ ]:
# Identificar valores extremos de pm25
Q1 = df['pm25'].quantile(0.25)
Q3 = df['pm25'].quantile(0.75)
IQR = Q3 - Q1

limite = Q3 + 1.5 * IQR

atipicos = df[df['pm25'] > limite]

# Mostrar información relacionada
display(atipicos[['estacion', 'fecha', 'pm25', 'pm10']
                 ].sort_values('pm25', ascending=False)
)

,estacion,fecha,pm25,pm10
68,LOS ANGELES ORIENTE,2026-04-27,586.7,126.7
167,TEMUCO PADRE LAS CASAS,2026-06-10,544.4,113.7
344,LOS ANGELES ORIENTE,2026-06-22,499.9,122.2
151,COYHANCO,2026-04-11,497.0,NaN
220,OSORNO ENTRE LAGOS,2026-04-21,477.3,112.0
699,LOS ANGELES ORIENTE,2026-05-21,446.4,113.6
174,KINGSTON,2026-04-14,421.0,78.0
333,VALDIVIA CENTRO,2026-05-01,358.9,105.3
585,OSORNO ENTRE LAGOS,2026-04-12,329.7,97.0
511,TALCAHUANO,2026-04-24,292.2,78.7


Tu conclusión sobre los atípicos:\
+++++ Al cruzar los valores atípicos de pm25 con las estaciones, fechas y mediciones de pm10, se observa que los valores extremos aparecen en distintas estaciones y fechas, sin una concentración temporal evidente.

Se identificaron mediciones muy elevadas de pm25, alcanzando un máximo de 586,7. Además, en varios registros, pm25 supera considerablemente a pm10, lo que representa una posible inconsistencia, ya que PM2.5 corresponde a una fracción del material particulado PM10.

Por lo tanto, estos valores podrían corresponder a errores de medición o registro, aunque no es posible confirmarlo con la información disponible.

Se decide mantener los registros y declarar estos valores como una limitación del análisis, hasta contar con información que permita determinar si corresponden a errores o fenómenos reales de contaminación.\
En la tabla resultante, 12 de los 14 valores extremos mostrados de pm25 superan sus correspondientes valores de pm10; otro registro tiene pm10 faltante. Esto refuerza la necesidad de revisar la validez de esas mediciones antes de utilizarlas en el análisis.



---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

+++++++ Para este conjunto de datos de calidad del aire, se propone crear una variable llamada mes, a partir de la columna fecha.

Esta variable permitirá analizar cómo cambian las concentraciones de pm25 entre abril, mayo y junio de 2026.

In [ ]:
# Tu código aquí
# Crear una nueva variable a partir de fecha
df['mes'] = df['fecha'].dt.month  #este dt accede al mes (month) de cada fecha

# Mostrar los resultados
display(df[['estacion', 'fecha', 'mes', 'pm25']].head(10))

,estacion,fecha,mes,pm25
0,LOS ANGELES ORIENTE,2026-06-26,6,77.1
1,CORONEL NORTE,2026-04-05,4,31.6
2,OSORNO ENTRE LAGOS,2026-06-21,6,63.5
3,KINGSTON,2026-05-31,5,48.1
4,TEMUCO PADRE LAS CASAS,2026-05-22,5,85.0
5,TALCAHUANO,2026-06-16,6,41.3
6,OSORNO ENTRE LAGOS,2026-05-01,5,NaN
7,CORONEL NORTE,2026-05-31,5,52.0
8,LOS ANGELES ORIENTE,2026-05-12,5,77.6
9,TEMUCO PADRE LAS CASAS,2026-06-18,6,63.9


Se utilizará  esta nueva variable mes en el análisis.
Podemos calcular el promedio de pm25 por me:

In [ ]:
tabla = df.groupby('mes')['pm25'].mean().round(2)
#Esto permitirá identificar qué mes presenta mayores concentraciones promedio de material particulado fino.
display(tabla)

,pm25
mes,
4,61.21
5,59.81
6,65.99


Por qué esta variable aporta a mi pregunta:\
+++++ La variable mes permite agrupar las mediciones de calidad del aire por períodos y analizar cómo varían las concentraciones de PM2.5 a lo largo del tiempo.

Esto facilita comparar los niveles de contaminación entre abril, mayo y junio, identificar posibles tendencias temporales y determinar en qué meses se presentan mayores concentraciones.


### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [ ]:
# Tu código aquí. Referencia:
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# df_cod = pd.get_dummies(df, columns=['TU_COLUMNA_CATEGORICA'], drop_first=True)


+++++ En este conjunto de datos hay variables numéricas (pm25, pm10, temperatura_c, humedad_pct) y categóricas (estacion, comuna, region, validado).\
Para esta actividad es posible aplicar dos transformaciones:
1. Normalización: llevar las variables numéricas a una escala común.
2. Codificación: transformar variables categóricas en columnas numéricas.

In [ ]:
#Se normalizarán las variables numéricas
#Se utilizará MinMaxScaler, que transforma los valores a una escala entre 0 y 1
from sklearn.preprocessing import MinMaxScaler

# Seleccionar columnas numéricas
columnas = ['pm25', 'pm10', 'temperatura_c', 'humedad_pct']

# Crear el normalizador
scaler = MinMaxScaler()

# Crear copia para conservar los datos originales
df_norm = df.copy()

# Normalizar
df_norm[columnas] = scaler.fit_transform(df[columnas])

# Mostrar resultado
display(df_norm[columnas].head(10))

,pm25,pm10,temperatura_c,humedad_pct
0,0.111886,0.739787,0.458537,0.157350
1,0.032590,0.163962,0.614634,0.304348
2,0.088184,0.496363,0.321951,0.668737
3,0.061345,0.317292,0.697561,0.505176
4,0.125654,0.780078,0.556098,0.666667
5,0.049495,0.344152,0.317073,0.782609
6,NaN,0.605484,0.678049,0.111801
7,0.068142,0.459429,0.531707,0.426501
8,0.112757,0.635702,0.692683,0.811594
9,0.088881,0.489647,0.448780,0.890269


+++++ En este conjunto de datos existen valores extremos de pm25. MinMaxScaler es sensible a ellos, por lo que conviene conservar los valores originales y evaluar el efecto de la transformación.

++++ Se codificará la variable categórica estacion utilizando pd.get_dummies().

In [ ]:
# Codificar estaciones
df_cod = pd.get_dummies(
    df,
    columns=['estacion'],
    drop_first=True,
    dtype=int
)

# Mostrar columnas creadas
display(df_cod.filter(like='estacion_').head(10))


,estacion_id,estacion_COYHANCO,estacion_KINGSTON,estacion_LOS ANGELES ORIENTE,estacion_OSORNO ENTRE LAGOS,estacion_TALCAHUANO,estacion_TEMUCO PADRE LAS CASAS,estacion_VALDIVIA CENTRO
0,2,0,0,1,0,0,0,0
1,8,0,0,0,0,0,0,0
2,6,0,0,0,1,0,0,0
3,1,0,1,0,0,0,0,0
4,5,0,0,0,0,0,1,0
5,4,0,0,0,0,1,0,0
6,6,0,0,0,1,0,0,0
7,8,0,0,0,0,0,0,0
8,2,0,0,1,0,0,0,0
9,5,0,0,0,0,0,1,0


+++++++ Se prepararon las variables numéricas y categóricas para futuros análisis. Las mediciones de calidad del aire se normalizaron mediante MinMaxScaler, llevándolas a una escala común entre 0 y 1.

Además, se codificó la variable estación mediante get_dummies, transformando sus categorías en columnas numéricas para usarlas en futuros modelos que requieran números.

Estas transformaciones permiten preparar el conjunto de datos para aplicar métodos estadísticos y modelos de aprendizaje automático, conservando los datos originales para futuras comparaciones.

---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [ ]:
def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """Flujo completo de limpieza de mi conjunto. Devuelve (df_limpio, informe)."""
    np.random.seed(semilla)
    d = pd.read_csv(ruta, sep=sep)
    informe = {'filas_iniciales': len(d)}

    # 1 · centinelas
    # 2 · categorías
    # 3 · duplicados
    # 4 · tipos
    # 5 · atípicos
    # 6 · atributos derivados

    informe['filas_finales'] = len(d)
    return d, informe


limpio, informe = limpiar(RUTA + ARCHIVO)
print(informe)

{'filas_iniciales': 734, 'filas_finales': 734}


In [ ]:

def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """Flujo completo de limpieza. Devuelve (df_limpio, informe)."""

    np.random.seed(semilla)
    d = pd.read_csv(ruta, sep=sep)
    informe = {'filas_iniciales': len(d)}

    # 1 · Centinelas
    d['pm10'] = pd.to_numeric(
        d['pm10'].astype(str).str.replace(',', '.'),
        errors='coerce'
    )
    informe['centinelas_pm10'] = (d['pm10'] == -999).sum()
    d['pm10'] = d['pm10'].replace(-999, np.nan)

    # 2 · Categorías
    columnas = ['estacion', 'comuna', 'region', 'validado']

    for col in columnas:
        d[col] = normalizar_texto(d[col])

    equivalencias = {'S': 'SI', 'N': 'NO'}
    d['validado'] = d['validado'].replace(equivalencias)

    # 3 · Duplicados
    informe['duplicados'] = d.duplicated().sum()
    d = d.drop_duplicates().copy()

    # 4 · Tipos
    columnas = ['pm25', 'pm10', 'temperatura_c', 'humedad_pct']

    for col in columnas:
        d[col] = pd.to_numeric(
            d[col].astype(str).str.replace(',', '.'),
            errors='coerce'
        )

    fechas = d['fecha'].astype(str).str.strip()

    fecha1 = pd.to_datetime(
        fechas, format='%Y-%m-%d', errors='coerce'
    )
    fecha2 = pd.to_datetime(
        fechas, format='%d/%m/%Y', errors='coerce'
    )

    d['fecha'] = fecha1.fillna(fecha2)

    # 5 · Atípicos
    informe['atipicos'] = {}

    for col in columnas:
        Q1 = d[col].quantile(0.25)
        Q3 = d[col].quantile(0.75)
        IQR = Q3 - Q1

        inferior = Q1 - 1.5 * IQR
        superior = Q3 + 1.5 * IQR

        cantidad = (
            (d[col] < inferior) | (d[col] > superior)
        ).sum()

        informe['atipicos'][col] = int(cantidad)

    # Los atípicos se conservan

    # 6 · Atributos derivados
    d['mes'] = d['fecha'].dt.month

    # Informe final
    informe['faltantes'] = d.isna().sum().to_dict()
    informe['filas_finales'] = len(d)

    return d, informe


limpio, informe = limpiar(RUTA + ARCHIVO)

print(informe)
display(limpio.head(10))

{'filas_iniciales': 734, 'centinelas_pm10': np.int64(18), 'duplicados': np.int64(14), 'atipicos': {'pm25': 16, 'pm10': 5, 'temperatura_c': 3, 'humedad_pct': 1}, 'faltantes': {'id_registro': 0, 'estacion_id': 0, 'estacion': 0, 'comuna': 0, 'region': 0, 'fecha': 0, 'pm25': 34, 'pm10': 18, 'temperatura_c': 0, 'humedad_pct': 0, 'validado': 0, 'mes': 0}, 'filas_finales': 720}


,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado,mes
0,02-20260626,2,LOS ANGELES ORIENTE,LOS ANGELES,BIOBIO,2026-06-26,77.1,154.9,9.5,58.3,SI,6
1,08-20260405,8,CORONEL NORTE,CORONEL,BIOBIO,2026-04-05,31.6,52.0,12.7,65.4,SI,4
2,06-20260621,6,OSORNO ENTRE LAGOS,OSORNO,LOS LAGOS,2026-06-21,63.5,111.4,6.7,83.0,SI,6
3,01-20260531,1,KINGSTON,CONCEPCION,BIOBIO,2026-05-31,48.1,79.4,14.4,75.1,SI,5
4,05-20260522,5,TEMUCO PADRE LAS CASAS,TEMUCO,ARAUCANIA,2026-05-22,85.0,162.1,11.5,82.9,SI,5
5,04-20260616,4,TALCAHUANO,TALCAHUANO,BIOBIO,2026-06-16,41.3,84.2,6.6,88.5,SI,6
6,06-20260501,6,OSORNO ENTRE LAGOS,OSORNO,LOS LAGOS,2026-05-01,NaN,130.9,14.0,56.1,SI,5
7,08-20260531,8,CORONEL NORTE,CORONEL,BIOBIO,2026-05-31,52.0,104.8,11.0,71.3,SI,5
8,02-20260512,2,LOS ANGELES ORIENTE,LOS ANGELES,BIOBIO,2026-05-12,77.6,136.3,14.3,89.9,SI,5
9,05-20260618,5,TEMUCO PADRE LAS CASAS,TEMUCO,ARAUCANIA,2026-06-18,63.9,110.2,9.3,93.7,SI,6


In [ ]:
# Comprobación: ejecuta la función DOS VECES y verifica que da lo mismo.
a, _ = limpiar(RUTA + ARCHIVO)
b, _ = limpiar(RUTA + ARCHIVO)
print('¿Reproducible?', a.equals(b))

¿Reproducible? True


In [ ]:
# Guarda el conjunto limpio. Este archivo es el insumo de la semana 3.
limpio.to_csv(RUTA + 'mi_conjunto_limpio.csv', sep=';', index=False)
print('Guardado:', limpio.shape)

Guardado: (720, 12)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:** _…_

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_